# 04 — Sales Streaming → Iceberg (with Customer Enrichment, via Kafka)

**Pre-requisite:** Start the sales producer in a separate terminal:
```bash
# host machine (Kafka on localhost:29092)
uv run python sales_producer.py

# or inside Docker
KAFKA_BOOTSTRAP_SERVERS=kafka:9092 uv run python sales_producer.py
```

**What this notebook does:**
1. Loads `app/data/source/customers.csv` as a **static** broadcast reference
2. Creates Iceberg table `iceberg_catalog.my_database.sales_enriched`
3. Reads a stream of JSON messages from Kafka topic `sales-events`
4. Enriches each sale event with customer info via a **stream-static join**
5. Writes the enriched stream into Iceberg (append mode)
6. Lets you monitor row count and revenue in real time

> The producer publishes 2 events every 10 s.  
> Spark triggers every 15 s so you will see rows accumulate in batches.

**Schema after enrichment:**

| Column | Source |
|--------|--------|
| sale_id, customer_id, product_id, product_name, category, quantity, unit_price, total_amount, store_id, payment_method, sale_timestamp | sales stream |
| customer_name, email, city, state, membership_tier | customers.csv join |

In [ ]:
from app.utils.spark_session import spark, display_df
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Kafka address: resolved by Spark Connect server inside Docker network
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS}")

spark

In [18]:
display(customers_df)

customer_id,name,email,phone,city,state,membership_tier,join_date
C001,Alice Johnson,alice.johnson@exa...,555-0101,San Francisco,CA,Gold,2022-01-15
C002,Bob Smith,bob.smith@example...,555-0102,New York,NY,Silver,2021-06-20
C003,Carol White,carol.white@examp...,555-0103,Chicago,IL,Bronze,2023-03-10
C004,David Lee,david.lee@example...,555-0104,Austin,TX,Gold,2020-11-05
C005,Eve Martinez,eve.martinez@exam...,555-0105,Seattle,WA,Platinum,2019-08-22
C006,Frank Brown,frank.brown@examp...,555-0106,Boston,MA,Silver,2022-07-18
C007,Grace Kim,grace.kim@example...,555-0107,Los Angeles,CA,Gold,2021-02-28
C008,Henry Wilson,henry.wilson@exam...,555-0108,Denver,CO,Bronze,2023-09-01
C009,Iris Chen,iris.chen@example...,555-0109,Portland,OR,Silver,2022-04-14
C010,James Davis,james.davis@examp...,555-0110,Miami,FL,Bronze,2023-11-30


---
## Step 1 — Load static customer reference

Broadcasted to every executor so the stream-static join is efficient.

In [16]:
customers_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("./app/data/source/customers.csv")
)

customers_df.printSchema()
customers_df.show(truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- membership_tier: string (nullable = true)
 |-- join_date: date (nullable = true)

+-----------+-------------+-------------------------+--------+-------------+-----+---------------+----------+
|customer_id|name         |email                    |phone   |city         |state|membership_tier|join_date |
+-----------+-------------+-------------------------+--------+-------------+-----+---------------+----------+
|C001       |Alice Johnson|alice.johnson@example.com|555-0101|San Francisco|CA   |Gold           |2022-01-15|
|C002       |Bob Smith    |bob.smith@example.com    |555-0102|New York     |NY   |Silver         |2021-06-20|
|C003       |Carol White  |carol.white@example.com  |555-0103|Chicago      |IL   |Bronze         |2023-03-10|
|C004       |David Lee

---
## Step 2 — Create Iceberg target table

In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg_catalog.my_database.sales_enriched (
        sale_id         STRING,
        customer_id     STRING,
        product_id      STRING,
        product_name    STRING,
        category        STRING,
        quantity        INT,
        unit_price      DOUBLE,
        total_amount    DOUBLE,
        store_id        STRING,
        payment_method  STRING,
        sale_timestamp  STRING,
        customer_name   STRING,
        email           STRING,
        city            STRING,
        state           STRING,
        membership_tier STRING
    )
    USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")

print("Table ready: iceberg_catalog.my_database.sales_enriched")
spark.sql("DESCRIBE iceberg_catalog.my_database.sales_enriched").show()

Table ready: iceberg_catalog.my_database.sales_enriched
+---------------+---------+-------+
|       col_name|data_type|comment|
+---------------+---------+-------+
|        sale_id|   string|   NULL|
|    customer_id|   string|   NULL|
|     product_id|   string|   NULL|
|   product_name|   string|   NULL|
|       category|   string|   NULL|
|       quantity|      int|   NULL|
|     unit_price|   double|   NULL|
|   total_amount|   double|   NULL|
|       store_id|   string|   NULL|
| payment_method|   string|   NULL|
| sale_timestamp|   string|   NULL|
|  customer_name|   string|   NULL|
|          email|   string|   NULL|
|           city|   string|   NULL|
|          state|   string|   NULL|
|membership_tier|   string|   NULL|
+---------------+---------+-------+



---
## Step 3 — Define streaming schema and Kafka reader

In [4]:
CHECKPOINT_DIR = "./.tmp/checkpoint_sales_enriched"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

sale_schema = StructType([
    StructField("sale_id",        StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("product_name",   StringType(),  True),
    StructField("category",       StringType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("total_amount",   DoubleType(),  True),
    StructField("store_id",       StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("timestamp",      StringType(),  True),
])

# Read from Kafka — each message value is a JSON-encoded sale event
raw_stream = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
         .option("subscribe", "sales-events")
         .option("startingOffsets", "latest")
         .load()
)

# Decode binary value → parse JSON → flatten columns
stream_df = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), sale_schema).alias("data"))
    .select("data.*")
)

print("Is streaming:", stream_df.isStreaming)
stream_df.printSchema()

Is streaming: True
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- timestamp: string (nullable = true)



---
## Step 4 — Start streaming write with customer enrichment

Stream-static join: each micro-batch is left-joined against the broadcasted  
`customers_df` so we get customer info without re-reading CSV every batch.

> This cell returns immediately. The stream runs in the background.

In [5]:
enriched_df = (
    stream_df
    .join(F.broadcast(customers_df), on="customer_id", how="left")
    .select(
        "sale_id",
        "customer_id",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        "total_amount",
        "store_id",
        "payment_method",
        F.col("timestamp").alias("sale_timestamp"),
        F.col("name").alias("customer_name"),
        "email",
        "city",
        "state",
        "membership_tier",
    )
)

query = (
    enriched_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="15 seconds")
    .toTable("iceberg_catalog.my_database.sales_enriched")
)

print(f"Stream started  — id: {query.id}")
print(f"Status         : {query.status}")

Stream started  — id: 5ee154f8-3e44-46c6-994b-3222cdb5163f
Status         : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


---
## Step 5 — Monitor (re-run this cell every ~15–30 s)

In [16]:
print("Active :", query.isActive)
print("Status :", query.status)

if query.lastProgress:
    prog = query.lastProgress
    print(f"Batch  : {prog['batchId']}")
    print(f"Input  : {prog['numInputRows']} rows in last batch")

print()

# Row count
spark.sql("""
    SELECT COUNT(*) AS total_sales
    FROM iceberg_catalog.my_database.sales_enriched
""").show()

# Revenue by category
spark.sql("""
    SELECT
        category,
        COUNT(*)                          AS sales_count,
        ROUND(SUM(total_amount), 2)       AS total_revenue,
        ROUND(AVG(total_amount), 2)       AS avg_order_value
    FROM iceberg_catalog.my_database.sales_enriched
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

# Revenue by membership tier
spark.sql("""
    SELECT
        membership_tier,
        COUNT(DISTINCT customer_id)       AS unique_customers,
        COUNT(*)                          AS sales_count,
        ROUND(SUM(total_amount), 2)       AS total_revenue
    FROM iceberg_catalog.my_database.sales_enriched
    GROUP BY membership_tier
    ORDER BY total_revenue DESC
""").show()

Active : True
Status : {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
Batch  : 29
Input  : None rows in last batch

+-----------+
|total_sales|
+-----------+
|         86|
+-----------+

+-----------+-----------+-------------+---------------+
|   category|sales_count|total_revenue|avg_order_value|
+-----------+-----------+-------------+---------------+
|Electronics|         48|     58138.67|        1211.22|
|       Home|         16|      7409.55|          463.1|
|   Clothing|         13|      3839.68|         295.36|
|      Books|          9|       899.82|          99.98|
+-----------+-----------+-------------+---------------+

+---------------+----------------+-----------+-------------+
|membership_tier|unique_customers|sales_count|total_revenue|
+---------------+----------------+-----------+-------------+
|         Silver|               6|         30|     26819.29|
|           Gold|               6|         28|     24929.19|
|       Platinu

---
## Step 6 — Stop the stream

In [ ]:
query.stop()
print("Stream stopped.")

# Final summary
spark.sql("""
    SELECT
        COUNT(*)                    AS total_sales,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT product_id)  AS unique_products,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM iceberg_catalog.my_database.sales_enriched
""").show()

In [13]:
%%sparksql
select count(1) FROM iceberg_catalog.my_database.sales_enriched;

active spark session is not found


In [14]:
%load_ext sparksql_magic

The sparksql_magic extension is already loaded. To reload it, use:
  %reload_ext sparksql_magic


In [12]:
spark